# **Regression Based Betting Algorithm**

In [3]:
import pandas as pd
# Load and merge all processed data
df_1 = pd.read_csv('../data/hist-data/processed/E0-2024-2025.csv')
df_2 = pd.read_csv('../data/hist-data/processed/E0-2023-2024.csv')
df_3 = pd.read_csv('../data/hist-data/processed/E0-2022-2023.csv')
df_4 = pd.read_csv('../data/hist-data/processed/E0-2021-2022.csv')
df_5 = pd.read_csv('../data/hist-data/processed/E0-2020-2021.csv')

# Merge all dataframes
df = pd.concat([df_1, df_2, df_3, df_4, df_5])

# Save merged dataframe
df.sort_values(by='Date', inplace=True, ascending=True)

,Date,Time,HomeTeam,AwayTeam,FTR,B365H,B365D,B365A,HomeWin,AwayWin,Draw,HomePoints,HomePPG_Last5
379,2020-09-12,17:30,Liverpool,Leeds,H,1.28,6.00,9.50,1,0,0,3,3.0
376,2020-09-12,20:00,West Ham,Newcastle,A,2.15,3.40,3.40,0,1,0,0,0.0
378,2020-09-12,15:00,Crystal Palace,Southampton,H,3.10,3.25,2.37,1,0,0,3,3.0
377,2020-09-12,12:30,Fulham,Arsenal,A,6.00,4.33,1.53,0,1,0,0,0.0
375,2020-09-13,16:30,Tottenham,Everton,A,1.83,3.60,4.33,0,1,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
253,2025-04-02,19:45,Southampton,Crystal Palace,D,5.75,3.80,1.62,0,0,1,1,0.2
74,2025-04-02,19:45,Brighton,Aston Villa,A,2.05,3.80,3.25,0,1,0,0,1.8
223,2025-04-02,19:45,Newcastle,Brentford,H,1.73,4.50,4.00,1,0,0,3,1.8
179,2025-04-02,20:00,Liverpool,Everton,H,1.36,5.00,8.00,1,0,0,3,3.0


In [4]:
# Split data into training and testing sets
train_df = df[df['Date'] < '2024-08-01']
test_df = df[df['Date'] >= '2024-08-01']

In [5]:
# Regress HomeWin on HomePPG_Last5 with a binary outcome
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(train_df[['HomePPG_Last5']], train_df['HomeWin'])

# Predict HomeWin on test set
test_df['HomeWin_Pred'] = model.predict(test_df[['HomePPG_Last5']])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(test_df['HomeWin'], test_df['HomeWin_Pred'])
print(f"Accuracy: {accuracy}")



Accuracy: 0.7424749163879598


/var/folders/s4/nk53mvwj21s9smsbw7lnc9v80000gn/T/ipykernel_73623/379091197.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['HomeWin_Pred'] = model.predict(test_df[['HomePPG_Last5']])


In [6]:
test_df

,Date,Time,HomeTeam,AwayTeam,FTR,B365H,B365D,B365A,HomeWin,AwayWin,Draw,HomePoints,HomePPG_Last5,HomeWin_Pred
195,2024-08-16,20:00,Man United,Fulham,H,1.60,4.20,5.25,1,0,0,3,3.0,1
105,2024-08-17,15:00,Everton,Brighton,A,2.63,3.30,2.63,0,1,0,0,0.0,0
135,2024-08-17,12:30,Ipswich,Liverpool,A,8.50,5.50,1.33,0,1,0,0,0.0,0
210,2024-08-17,15:00,Newcastle,Southampton,H,1.36,5.25,8.00,1,0,0,3,3.0,1
224,2024-08-17,15:00,Nott'm Forest,Bournemouth,D,2.45,3.50,2.80,0,0,1,1,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253,2025-04-02,19:45,Southampton,Crystal Palace,D,5.75,3.80,1.62,0,0,1,1,0.2,0
74,2025-04-02,19:45,Brighton,Aston Villa,A,2.05,3.80,3.25,0,1,0,0,1.8,1
223,2025-04-02,19:45,Newcastle,Brentford,H,1.73,4.50,4.00,1,0,0,3,1.8,1
179,2025-04-02,20:00,Liverpool,Everton,H,1.36,5.00,8.00,1,0,0,3,3.0,1


In [8]:
def calculate_profit(row):
    if row['HomeWin_Pred'] == 1:
        if row['HomeWin'] == 1:
            return row['B365H'] - 1  # profit = payout - stake
        else:
            return -1  # lose the stake
    else:
        return 0  # no bet placed

test_df['Profit'] = test_df.apply(calculate_profit, axis=1)

# Total profit and ROI
total_profit = test_df['Profit'].sum()
total_bets = test_df['HomeWin_Pred'].sum()  # count of 1s in PredHomeWin
roi = (total_profit / total_bets) * 100 if total_bets > 0 else 0

print(f"Total profit: ${total_profit:.2f}")
print(f"Total bets placed: {total_bets}")
print(f"ROI: {roi:.2f}%")


Total profit: $34.62
Total bets placed: 124
ROI: 27.92%


/var/folders/s4/nk53mvwj21s9smsbw7lnc9v80000gn/T/ipykernel_73623/4250833585.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['Profit'] = test_df.apply(calculate_profit, axis=1)
